# Job 49816217 - narrowing the MLP at N=96, and L* vs (N, ff)

**Headline.** At N=96, narrowing `dim_feedforward` from 1024 to 256 costs **7.1 points** on the
shared 1500-instance pool (0.8407 vs 0.9113, McNemar exact p = 2.3e-11), and adding four more
layers to the thin model makes it **worse still** (0.8100, -10.1 vs the wide knee, p = 8.4e-19).
L=12/ff=256 clears **none** of the six rho=0.60 depth bins, so `L*(96, ff=256) > 12`: the width
cost is super-multiplicative in N, which is the falsifier I named for pre-registration (b).
The L=12 failure is, however, **confounded by trainability** - its well-formedness collapses
while its conditional deduction stays near the wide knee's (section 3).

**Provenance**

| | |
|---|---|
| Slurm job | 49816217, `COMPLETED`, ExitCode 0:0, elapsed 15:50:15, ended 2026-10-28T02:30:07 |
| script | `scripts/slurm/sudoku2_n96_ff256_lstar.sbatch` |
| git commit at submit | `3873723` (submitted 2026-10-27T10:34:20) |
| seed | 0 (both trained cells) |
| instances | N=96, degree 5, q=4, rho 0.30-0.95, removal hints, **shuffle_adj=False** |
| architecture | d_model 256, 4 heads, pre-norm, lr=1e-3 constant, batch 32 |
| cells | L=8/ff=256 and L=12/ff=256, both vs finding 22's L=8/ff=1024 knee |
| run dir | `$SCRATCH/spin_gnn/sudoku_n96_ff256_2026-10-27` |
| reference run | `$SCRATCH/spin_gnn/sudoku_n96_layer_scan_2026-10-26` (job 49639312) |
| deck | `notebooks/sudoku_n96_ff256_2026-10-28/ff256.pdf` (9 pages) |

**Every number below is a DECODING score.** `shuffle_adj=False`, so each node's adjacency row
still arrives sorted by the neighbour's true colour - finding 16's leak. That is deliberate:
this run extends the leaked-order L* ladder (findings 14, 18, 22) rather than starting a second
one, and the previous best on this eval is finding 22's L=8/ff=1024, which is row 0 of every
table here because it was re-scored by the same `agg_eval` call.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

SCRATCH = os.environ["SCRATCH"]
RUN = f"{SCRATCH}/spin_gnn/sudoku_n96_ff256_2026-10-27"
WIDE = f"{SCRATCH}/spin_gnn/sudoku_n96_layer_scan_2026-10-26"
pd.set_option("display.width", 200)

# (label, checkpoint substring, colour), reference first - the order agg_eval was given.
CELLS = [("L=8, ff=1024", "L08_ff01024", "#2ca02c"),
         ("L=8, ff=256", "L08_ff00256", "#ff7f0e"),
         ("L=12, ff=256", "L12_ff00256", "#d62728")]
NAMES = [c[0] for c in CELLS]
GATE = 0.90

pool = pd.read_csv(f"{RUN}/sharedpool_n1500.csv")
pool["cell"] = NAMES
# The thin cells' depth eval is in this run; the wide knee's is in the layer scan, from the
# same generator stream and the same --n-pool 600 --min-per-bin 40, so bins line up row by row.
wide = pd.read_csv(f"{WIDE}/depth_by_layers_n96.csv")
depth = pd.concat([wide[wide.checkpoint.str.contains("L08_ff01024")],
                   pd.read_csv(f"{RUN}/depth_by_layers_n96_ff256.csv")], ignore_index=True)
depth["k"] = (depth.solve_rate * depth.n).round().astype(int)
sweep = pd.concat([pd.read_csv(f"{WIDE}/sweep.csv"), pd.read_csv(f"{RUN}/sweep.csv")],
                  ignore_index=True).set_index(["num_layers", "dim_feedforward"])


def bins60(tag, rho=0.60):
    return depth[depth.checkpoint.str.contains(tag)].query("rho_requested == @rho").sort_values("T")


def ca_trend(T, k, n):
    """Cochran-Armitage trend across ordered depth bins (as notebooks.make_n96_lstar_deck)."""
    T, k, n = np.asarray(T, float), np.asarray(k, float), np.asarray(n, float)
    N, K = n.sum(), k.sum()
    p = K / N
    Tbar = (n * T).sum() / N
    z = (k * (T - Tbar)).sum() / np.sqrt(p * (1 - p) * (n * (T - Tbar) ** 2).sum())
    return z, 2 * stats.norm.sf(abs(z))


print(len(pool), "pool rows;", len(depth), "depth rows;", len(sweep), "sweep rows")

## 1. Metric of record: the shared n=1500 pool

`sudoku_agg_eval` builds one pool at `seed_start=50000000000` - far above every training seed -
and scores all three checkpoints on it. All three share `shuffle_adj=False` and differ only in
architecture, so the prompts are byte-identical and the test is a McNemar exact binomial on the
discordant pairs. The per-run 300-instance `agg_solve_rate` in `sweep.csv` is retired and is not
quoted anywhere below.

In [ ]:
cols = ["cell", "solve_rate", "wilson_lo", "wilson_hi", "delta", "discordant_won",
        "discordant_lost", "mcnemar_exact_p", "truncated_rate", "outcome_accuracy"]
display(pool[cols].round(4))

# Architecture and teacher-forced loss, pulled per cell so nothing is hardcoded.
rows = []
for name, tag, _ in CELLS:
    L, ff = int(tag.split("_")[0][1:]), int(tag.split("ff")[1])
    r = sweep.loc[(L, ff)]
    pr = pool[pool.checkpoint.str.contains(tag)].iloc[0]
    rows.append({"cell": name, "n_params": int(r.n_params),
                 "vs_wide_knee": round(r.n_params / sweep.loc[(8, 1024)].n_params, 3),
                 "max_len": int(r.max_len), "val_nll": round(r.final_val_nll, 4),
                 "val_acc": round(r.final_val_accuracy, 4),
                 "pool_solve_rate": round(pr.solve_rate, 4)})
display(pd.DataFrame(rows))

z = np.load(f"{RUN}/sharedpool_n1500_per_instance.npz", allow_pickle=True)
s = z["solved"]
print("solved per checkpoint:", s.sum(1), "of", s.shape[1], "| seed_start", z["seed_start"])
print("L=8/ff=256 vs L=12/ff=256 discordant:",
      int((s[1] & ~s[2]).sum()), "won /", int((~s[1] & s[2]).sum()), "lost")

### Two things to read off that table

**It is not a parameter story.** L=12/ff=256 has 5.01M parameters against the wide knee's 6.58M
(0.76x), but L=8/ff=256 has only 3.43M and beats it by 3.1 points. Inside the thin column,
+1.58M parameters spent on depth bought a *loss*. Finding 13's "depth is the cheaper currency"
does not survive to N=96.

**`val_nll` ranks the cells almost in reverse of `solve_rate`.** The worst cell on the pool
(0.8100) has the *best* teacher-forced nll (1.4108). Next-token loss and free generation come
apart here, which section 3 explains: nothing in this project should be ranked on `val_nll`.

**A free replication check.** The reference checkpoint was scored by two independent `agg_eval`
calls a day apart - once in job 49639312's pool and once here - and both report 0.911333 on
1500 instances. Same `seed_start`, same `INSTANCE_KEYS`, deterministic generation. That is a
check on the harness, not a result.

## 2. Six depth bins, and where L* is read off

L* is the smallest `num_layers` reaching 0.90 on depth-T instances at fixed *realized* density.
`sudoku_depth_eval` bins a 600-instance pool by DD depth T; at rho=0.60 the realized density is
0.594 in every bin and the bins are identical row by row across the three columns, so the
columns compare bin by bin.

In [ ]:
for rho in (0.60, 0.75):
    g0 = bins60("L08_ff01024", rho)
    if not len(g0):
        continue
    t = pd.DataFrame({name: bins60(tag, rho).set_index("T").solve_rate.round(3)
                      for name, tag, _ in CELLS})
    t["n"] = g0.set_index("T").n
    print(f"=== rho={rho}, realized density {g0.realized_rho.round(4).unique()}")
    display(t)

for name, tag, _ in CELLS:
    g = bins60(tag)
    zc, p = ca_trend(g["T"], g.k, g.n)
    print(f"{name:<14} bins >= {GATE}: {int((g.solve_rate >= GATE).sum())} of {len(g)}"
          f" | Cochran-Armitage z={zc:+.2f}, p={p:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.4))
for name, tag, col in CELLS:
    g = bins60(tag)
    lo, hi = [], []
    for _, r in g.iterrows():
        a, b = stats.beta.interval(0.95, r.k + 0.5, r.n - r.k + 0.5)
        lo.append(max(0.0, r.solve_rate - a))   # a bin at exactly 1.0 would give a negative yerr
        hi.append(max(0.0, b - r.solve_rate))
    ax.errorbar(g["T"], g.solve_rate, yerr=[lo, hi], fmt="-o", color=col, ms=5, lw=1.8,
                capsize=2, label=name)
ax.axhline(GATE, color="0.35", ls="--", lw=1.2)
ax.set_xlabel("DD depth T"); ax.set_ylabel("solve_rate")
ax.set_title("rho=0.60, realized density 0.594 in every bin")
ax.set_xticks(range(4, 10)); ax.legend(fontsize=9, loc="lower left")
fig.tight_layout()

**The two thin cells fail in opposite shapes.** L=8/ff=256 declines with T the way a
capacity-limited reasoner should - clearing the gate at T=4 and T=5, falling to 0.812 by T=7,
CA p=0.026. L=12/ff=256 is flat (0.829-0.868 across the whole range) and its trend is
*positive*, z=+0.67. A model equally bad at T=4 and T=9 is not failing at deduction.

**The T=9 bin is not a recovery.** Both thin cells jump at T=9 (0.953 and 0.860). That bin has
n=43, the smallest in the sweep, and the same non-monotone bounce appears in the wide column of
finding 22. Deep instances surviving the `--min-per-bin` filter are selected for being
generable, so the last bin is not a like-for-like extension of the others; nothing here rests
on it.

**The resulting grid.** L* vs (N, ff), with this run's cell last:

| | L* at ff=1024 | L* at ff=256 | width factor |
|---|---|---|---|
| N=24 | 3 | not measured | -- |
| N=48 | 4 | 6 | 1.5x (finding 14) |
| N=96 | 8 | **> 12** | **> 1.5x (this run)** |

The rows are *not* paired with each other: different N is a different instance distribution and
a different vocabulary, so the width-factor column compares two knees each measured on its own
ladder and carries no p-value.

## 3. Mechanism: the deep-thin cell stopped writing valid traces

`well_formed_rate` is the fraction of generations that parse as a trace at all. Dividing
`solve_rate` by it gives the solve rate *conditional* on being well formed: how good the
deduction is once the grammar is out of the way. This is the analysis that was not
pre-registered and is the most interesting thing the run produced.

In [ ]:
rows = []
for name, tag, _ in CELLS:
    g = bins60(tag)
    cond = g.solve_rate / g.well_formed_rate
    rows.append({"cell": name,
                 "worst_well_formed": round(g.well_formed_rate.min(), 3),
                 "best_well_formed": round(g.well_formed_rate.max(), 3),
                 "solve|wf T=4": round(cond.iloc[0], 3),
                 "solve|wf T=9": round(cond.iloc[-1], 3),
                 "r(T, solve|wf)": round(float(np.corrcoef(g["T"], cond)[0, 1]), 2),
                 "r over T=4..7": round(float(np.corrcoef(g["T"][:4], cond[:4])[0, 1]), 2)})
display(pd.DataFrame(rows))

fig, ax = plt.subplots(figsize=(7.2, 4.4))
for name, tag, col in CELLS:
    g = bins60(tag)
    ax.plot(g["T"], g.solve_rate / g.well_formed_rate, "-o", color=col, ms=5, lw=1.8, label=name)
    ax.plot(g["T"], g.well_formed_rate, "--s", color=col, ms=4, lw=1.2, alpha=0.55)
ax.set_xlabel("DD depth T")
ax.set_ylabel("solid: solve | well formed\ndashed: well_formed_rate")
ax.set_xticks(range(4, 10)); ax.legend(fontsize=9, loc="lower left")
fig.tight_layout()

**L=12/ff=256 (red):** well-formedness collapses to 0.853, but once a generation parses it
solves 0.92-0.98 and *improves* with T (r=+0.72) - deduction near the wide knee's, output
malformed. The pool agrees the failure is generative: `truncated_rate` 0.0007 -> 0.0167 and
`outcome_accuracy` 0.986 -> 0.869 going from L=8 to L=12 at ff=256.

**L=8/ff=256 (orange):** the reverse. Well formed throughout (0.971-1.000), conditional rate
declining 0.976 -> 0.836 monotonically across T=4..7 (r=-1.00 over those four bins) - the
genuine depth-limited reasoner, and the cell that defines the width cost.

**Why this matters for reading L\*.** `L*(96, ff=256) > 12` is correct as stated - L=12 does not
clear the gate - but it is **not** evidence that 12 thin layers cannot do depth-9 deduction. At
ff=256 a 12-layer stack becomes hard to *train to emit the format*, and that confounds the knee
with trainability. This is why re-running L=12/ff=256 with warmup and a cosine schedule is the
top next step rather than a nicety.

## 4. Pre-registrations, as resolved

Written into `scripts/slurm/sudoku2_n96_ff256_lstar.sbatch` before the job ran.

| | claim | verdict | evidence |
|---|---|---|---|
| (a) | L=8/ff=256 does NOT clear all six rho=0.60 bins | **CONFIRMED** | clears 3 of 6 (T=4, 5, 9). Finding 14's width cost is present at N=96. |
| (b) | L=12/ff=256 DOES clear all six, i.e. `L*(96, ff=256) <= 12` | **FALSIFIED** | clears 0 of 6. The named falsifier - "a super-multiplicative width cost, which needs its own replication" - is what happened. |
| (c) | L=12/ff=256 is at least as good as L=8/ff=1024 at 0.76x parameters | **FALSIFIED** | 0.8100 vs 0.9113, -0.1013, p=8.4e-19. Resolvably worse. |
| (d) | Inside whichever thin column clears, L*(T) rises with T | **UNRESOLVABLE** | neither thin column clears, so the test has no subject. The L=12 trend is in fact positive (CA z=+0.67, p=0.503). |

**Three of four wrong, all in the same direction.** (b), (c) and the hypothesis in the sbatch
header all assumed the N=48 width factor would transfer. It did not. The pre-registered
falsifier for (b) names exactly what happened, which is the only reason this reads as a result
rather than a surprise. (d) is a design fault, not a prediction error: making it conditional on
a cell clearing means a two-cell job that fails twice cannot answer it at all.

L=16 was deliberately excluded from the grid - it would have pushed training past the eval
stage's start inside the job's time limit.

## 5. What this changes

- Every L* in STATUS.md keeps its "at ff=1024" caveat, and the caveat is now known to be
  load-bearing rather than cautious: at N=96 the thin knee is above 12 while the wide one is 8.
- Finding 13's "depth is the cheaper currency" is demoted to small N. At N=96 the extra layers
  did not merely fail to pay - they cost 3.1 points.
- The L* vs (N, ff) grid has five of six entries; the missing one (N=24 at ff=256) is the
  cheapest cell in the project to fill.
- `val_nll` is disqualified as a ranking statistic for this task: it ordered the three cells
  almost in reverse.

## 6. What it does not change

- These are **decoding** scores in the **leaked-order** task (`shuffle_adj=False`, finding 16).
  The whole L* ladder lives there, and job 49705064 has only just established the data budget a
  leak-free ladder would need (240000 instances x 100 epochs).
- The L=12 result is confounded by trainability, not just capacity (section 3), so
  "> 12" is a statement about *this training recipe* at ff=256.
- One seed per cell, and L* is read off a 0.90 gate on six bins - a coarse instrument that
  cannot distinguish L*=13 from L*=20.

## 7. Next, in order of information per GPU-hour

1. Re-run L=12/ff=256 with warmup and a cosine schedule. If `well_formed_rate` recovers to 0.98
   and the bins clear, the knee was trainability; if it stays flat, `L*(96, ff=256) > 12` is a
   statement about computation after all.
2. L=10/ff=1024 at N=96, to tighten the wide knee from "between 6 and 8" to one rung.
3. N=24 at ff=256 - hours, not a day - to get the width factor at the small end and see whether
   it grows with N monotonically.